# 🇸🇦 Arabic Piper TTS Fine-Tuning — Complete Pipeline

This notebook contains the **entire fine-tuning pipeline** for the Piper Arabic `ar_JO-kareem-medium` model.
Run each section top-to-bottom. If your Colab session disconnects, re-run from **Section 1** (setup is fast) and training auto-resumes from the last checkpoint on Google Drive.

### Sections:
1. ⚙️ Environment Setup & GPU Check
2. 📦 Dataset Download & Preparation
3. 🔊 Baseline Benchmark (Before Training)
4. 🏋️ Fine-Tuning & Checkpointing
5. 📊 Export, Evaluation & Comparison

---
## ⚙️ Section 1: Environment Setup

In [ ]:
# 1.1 Clone Repository from GitHub
# ⚠️ Replace YOUR_USERNAME with your actual GitHub username or org.
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/piper-tts-finetuning.git'  # <-- EDIT THIS
REPO_DIR = '/content/piper-tts-finetuning'

if os.path.exists(REPO_DIR):
    print(f'Repository already cloned. Pulling latest changes...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# 1.2 GPU Check
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1.3 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.4 Set up Drive Directories
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Arabic-Piper')
for s in ['datasets', 'processed', 'checkpoints', 'tensorboard', 'logs', 'outputs', 'metrics']:
    d = DRIVE_ROOT / s
    d.mkdir(parents=True, exist_ok=True)
    print(f'Verified: {d}')

In [ ]:
# 1.5 Install System & Python Dependencies
import sys; print(f'Python version: {sys.version}')
!apt-get update -q && apt-get install -y -q espeak-ng libespeak-ng-dev build-essential
!pip install -q -r requirements.txt
print('\n✅ Core dependencies installed.')

In [ ]:
# 1.6 Install Piper Training Engine & Build C-Extensions (piper_train)
import os

PIPER_SRC = '/content/piper'
if not os.path.exists(PIPER_SRC):
    !git clone https://github.com/rhasspy/piper.git {PIPER_SRC}

# Install piper_train without obsolete piper-phonemize dependency
!pip install -q cython setuptools "pytorch-lightning~=1.9.5"
!pip install -q --no-deps -e {PIPER_SRC}/src/python

# Compile monotonic_align using official build script
!cd {PIPER_SRC}/src/python && bash build_monotonic_align.sh
print('\n✅ piper_train installed & compiled.')

---
## 📦 Section 2: Dataset Download & Preparation

- **Cell 2.1**: Downloads the HuggingFace dataset and the base `.ckpt` + `config.json`.
- **Cell 2.2**: Converts audio to 22050 Hz WAVs, writes `metadata.csv`, copies `config.json` from the base checkpoint into the processed directory, and runs `piper_train.preprocess` to phonemize the text into `dataset.jsonl`. These two files are required by `piper_train`.

In [ ]:
# 2.1 Download Dataset & Base Checkpoint (includes config.json)
!python scripts/download_dataset.py --config configs/experiment001.yaml

In [ ]:
# 2.2 Prepare Dataset → wavs/ + metadata.csv + config.json + dataset.jsonl
# This step also runs piper_train.preprocess (phonemization). Expect it to take ~5-10 min.
!python scripts/prepare_dataset.py --config configs/experiment001.yaml

In [ ]:
# 2.3 Verify All Required Files Are Present
from pathlib import Path

processed_dir = Path('/content/drive/MyDrive/Arabic-Piper/processed/experiment001')
required = ['config.json', 'dataset.jsonl', 'metadata.csv', 'wavs']

print('Checking required piper_train inputs:')
all_ok = True
for name in required:
    p = processed_dir / name
    exists = p.exists()
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {name}')
    if not exists:
        all_ok = False

if all_ok:
    n_wavs = len(list((processed_dir / 'wavs').glob('*.wav')))
    n_jsonl = sum(1 for _ in open(processed_dir / 'dataset.jsonl', 'r'))
    print(f'\n✅ All inputs ready — {n_wavs} WAV files, {n_jsonl} phonemized entries.')
else:
    print('\n❌ Some files are missing. Re-run Cells 2.1 and 2.2.')

---
## 🔊 Section 3: Baseline Benchmark (Before Training)

In [ ]:
# 3.1 Download Base ONNX Model for Benchmarking
!mkdir -p /content/drive/MyDrive/Arabic-Piper/checkpoints/base/
!wget -q -O /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx \
    https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx
!wget -q -O /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx.json \
    https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/ar/ar_JO/kareem/medium/ar_JO-kareem-medium.onnx.json
print('✅ ONNX model downloaded.')

In [ ]:
# 3.2 Run Baseline Benchmark
!python scripts/benchmark.py \
    --model /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx \
    --model-config /content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar_JO-kareem-medium.onnx.json \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir /content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark

In [ ]:
# 3.3 Display Baseline Report & Play Sample
import json
from IPython.display import Audio, display, HTML
from pathlib import Path

report_file = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_report.json')
if report_file.exists():
    data = json.loads(report_file.read_text())
    print(f"Baseline Avg RTF: {data.get('avg_rtf')}")
    print(f"Total Audio Duration: {data.get('total_audio_duration_sec')}s")
    sample_wav = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_01.wav')
    if sample_wav.exists():
        display(HTML('<h4>🔊 Baseline Sample 1:</h4>'))
        display(Audio(str(sample_wav)))
else:
    print('Report not found — run Cell 3.2 first.')

---
## 🏋️ Section 4: Fine-Tuning & Checkpointing

Checkpoints save every 5 epochs to Google Drive. If session disconnects, re-run Section 1 — training resumes from the latest checkpoint automatically.

In [ ]:
# 4.1 Checkpoint Detection
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001')
ckpt_dir.mkdir(parents=True, exist_ok=True)

existing_ckpts = sorted(ckpt_dir.glob('*.ckpt'))
if existing_ckpts:
    resume_ckpt = str(existing_ckpts[-1])
    print(f'✅ Resuming from: {resume_ckpt}')
else:
    base_ckpt = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/base/ar/ar_JO/kareem/medium/epoch=5079-step=1682020.ckpt')
    resume_ckpt = str(base_ckpt) if base_ckpt.exists() else ''
    print(f'🆕 Starting from base checkpoint: {resume_ckpt or "(none — training from scratch)"}')

In [ ]:
# 4.2 Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001/lightning_logs

In [ ]:
# 4.3 Execute Training Run
# Note: PyTorch Lightning Trainer args use underscores (--max_epochs, --default_root_dir)
!python -m piper_train \
    --dataset-dir /content/drive/MyDrive/Arabic-Piper/processed/experiment001 \
    --accelerator gpu \
    --devices 1 \
    --batch-size 16 \
    --validation-split 0.05 \
    --max_epochs 50 \
    --checkpoint-epochs 5 \
    --default_root_dir /content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001 \
    --resume_from_checkpoint "{resume_ckpt}"

---
## 📊 Section 5: Export, Evaluation & Comparison

In [ ]:
# 5.1 Export Best Checkpoint to ONNX
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001')
best_ckpts = sorted(ckpt_dir.glob('*.ckpt'))

if best_ckpts:
    target_ckpt = str(best_ckpts[-1])
    output_onnx = '/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx'
    print(f'Exporting: {target_ckpt}')
    !python scripts/export_model.py --checkpoint "{target_ckpt}" --output-onnx "{output_onnx}"
else:
    print('No checkpoint found. Run training first.')

In [ ]:
# 5.2 Benchmark Fine-Tuned Model
!python scripts/benchmark.py \
    --model /content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir /content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark

In [ ]:
# 5.3 Side-by-Side Audio Comparison
from IPython.display import Audio, display, HTML
from pathlib import Path

base_wav = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_01.wav')
ft_wav   = Path('/content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark/benchmark_01.wav')

if base_wav.exists() and ft_wav.exists():
    display(HTML('<h3>🔊 Baseline (Before Training):</h3>'))
    display(Audio(str(base_wav)))
    display(HTML('<h3>🎙️ Fine-Tuned (After Training):</h3>'))
    display(Audio(str(ft_wav)))
else:
    print('Audio files not found. Complete training and benchmarking first.')

---
## ✅ Done!

Your fine-tuned ONNX model is saved at:
```
/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx
```

Test it locally:
```bash
python scripts/test_local.py --mode finetuned --model path/to/ar_JO_finetuned.onnx --text "السَّلَامُ عَلَيْكُمْ"
```